<a href="https://colab.research.google.com/github/Asheesh1272/Asheesh127/blob/main/Dive_Deep_Into_Titanic_and_get_accuracy_of_99_76_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Display plots directly in the notebook
%matplotlib inline
import seaborn as sns
sns.set_style('darkgrid')


# Import machine learning tools from scikit-learn

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict, cross_validate
# Linear models
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier, Perceptron

from sklearn.svm import SVC

from sklearn.ensemble import VotingClassifier
# K-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier
# Naive Bayes classifiers
from sklearn.naive_bayes import GaussianNB, MultinomialNB
# Decision Trees
from sklearn.tree import DecisionTreeClassifier
# Metrics for model evaluation
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report, confusion_matrix, roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score

In [ ]:

train = pd.read_csv(r'/content/train.csv')
test = pd.read_csv(r'/content/titanic_test.csv')
# Load the gender submission file (contains example predictions)
gender_sub = pd.read_csv(r'/content/gender_submission.csv')

In [ ]:
train.sample(5)

In [ ]:
from matplotlib import pyplot as plt
_df_17['Pclass'].plot(kind='line', figsize=(8, 4), title='Pclass')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
train['Age'] = round(train['Age'].fillna(train.Age.median(), inplace = False)).astype(int)
test['Age'] = round(test['Age'].fillna(test.Age.median(), inplace = False)).astype(int)

In [ ]:
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

In [ ]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

In [ ]:
train['Title'] = train['Name'].str.extract('([A-za-z]+)\.', expand = False)
test['Title'] = test['Name'].str.extract('([A-za-z]+)\.', expand = False)

In [ ]:
titles = list(set(train['Title']).union(set(test['Title'])))
print(titles)  # Display the complete list of unique titles

In [ ]:
# Define a list of titles that represent upper class passengers
upper_class = ['Jonkheer', 'Don', 'Countess', 'Dona', 'Sir', 'Rev', 'Capt', 'Major', 'Dr', 'Col', 'Lady']

# Create a list containing both training and test datasets for batch processing
titanic_data = [train, test]

# Process each dataset (train and test)
for data in titanic_data:
    # Standardize female titles: replace 'Mlle' and 'Miss' with 'Ms'
    data['Title'] = data['Title'].replace(['Mlle', 'Miss'], 'Ms')

    # Standardize married female titles: replace 'Mme' with 'Mrs'
    data['Title'] = data['Title'].replace('Mme', 'Mrs')


    data['Title'] = data['Title'].replace('Master', 'Mr')


    data['Title'] = data['Title'].replace(upper_class, 'Elite')

    # Create age bins by dividing age into 4 equal-sized groups (quartiles)
    # and label them with integers 1-4
    data['AgeBins'] = pd.qcut(data['Age'], 4, labels = [1,2,3,4])

In [ ]:
train['WealthState'] = pd.qcut(train['Fare'], 3, labels = ['poor', 'mediocre', 'rich'])
test['WealthState'] = pd.qcut(test['Fare'], 3, labels = ['poor', 'mediocre', 'rich'])

In [ ]:
train = train.drop(['PassengerId', 'Ticket', 'Cabin', 'Fare', 'Name', 'SibSp', 'Parch'], axis = 1)
test = test.drop(['PassengerId', 'Ticket', 'Cabin','Fare', 'Name', 'SibSp', 'Parch'], axis = 1)

In [ ]:
train.describe()

In [ ]:
train.info(), test.info()

In [ ]:
train.head()

In [ ]:
train.hist(figsize = (8,5), grid = False)
# Display the histograms
plt.show()


In [ ]:
# Define a list of features to analyze for survival correlation
train_grouping = ['Pclass','Sex', 'AgeBins','Embarked','Title','WealthState', 'FamilySize']

for x in train_grouping:

    train[[x, 'Survived']].groupby([x], as_index = False, observed = True).mean()

    # Print the feature name we're currently analyzing
    print('Survival Correlation by', x)

    print(train[[x, 'Survived']].groupby(x, as_index = False, observed = True).mean())

    # Print a separator line for better readability
    print('-' * 35)

In [ ]:
# Create a figure with specified size
plt.figure(figsize = (20,14))

plt.rcParams.update({'font.size': 12})
# Set transparency for plot elements
plotalpha = 0.8
# Get the viridis colormap
cmap = plt.get_cmap('viridis')
# Define two colors from the colormap
c1 = cmap(0.3)
c2 = cmap(0.7)

# List of features to analyze for survival correlation
traincols = ['Pclass', 'Sex', 'AgeBins','Embarked','Title','FamilySize','WealthState']

for i, data in enumerate(traincols, 1):
    rows = 3
    cols = 3
    plt.subplot(rows, cols, i)

    plt.hist(x = [train[train['Survived'] == 1][data], train[train['Survived'] == 0][data]],
             color = (c1, c2),
             alpha = plotalpha,
             label = ['Survived', 'Dead'])
    # Add title and labels
    plt.title(f'Survived by {data} Grouping')
    plt.xlabel(f'{data}')
    plt.ylabel('Number of Passengers')
    plt.legend()

# Adjust layout to prevent overlap
plt.tight_layout()
# Display the plot
plt.show()

In [ ]:
# Create a 2x2 grid of subplots with a figure size of 10x10 inches
fig, axes = plt.subplots(2, 2, figsize = (10,10))

plt.rcParams.update({'font.size': 12})
# Define the columns to analyze in the plots
trcols = ['AgeBins', 'Sex', 'Embarked', 'WealthState']
# Loop through each subplot and corresponding data column
for ax, data in zip(axes.flat, trcols):
    # For age bins and wealth state, create violin plots
    if data in ['AgeBins', 'WealthState']:
        sns.violinplot(x = train['Pclass'], y = data, hue = train['Survived'], data = train, ax = ax, orient = 'v', alpha= plotalpha)
        ax.set_title(f'Pclass {data} Survival Correlation')
        ax.legend(loc = 'upper right', title = 'Survived')
        ax.invert_yaxis()
    # For embarked port and sex, create strip plots
    elif data in ['Embarked', 'Sex']:
        sns.stripplot(x= train['Pclass'], y = data, hue = train['Survived'], data = train, ax = ax, orient = 'v', alpha = plotalpha)
        ax.set_title(f'Pclass {data} Survival correlation')
        ax.legend(loc = 'upper right', title = 'Survived')
        ax.invert_yaxis()

plt.tight_layout()
# Display the figure
plt.show()

In [ ]:
# Create a figure with 3 subplots in a row, with a size of 10x10 inches
fig, axes = plt.subplots(1, 3, figsize = (10,10))
# Increase font size for better readability
plt.rcParams.update({'font.size': 12})

sns.boxplot(x = 'Pclass', y = 'Age', hue = 'Survived', data = train, ax = axes[0], orient = 'v', boxprops=dict(alpha=0.8))
axes[0].set_title('Pclass vs Age Correlation')
ax.legend(loc = 'upper right', title = 'Survived')
ax.invert_yaxis()

sns.violinplot(x = 'Pclass', y = 'FamilySize', hue = 'Survived', data = train, ax = axes[1], orient = 'v', alpha = plotalpha)
axes[1].set_title('Pclass vs FamilySize Correlation')
ax.legend(loc = 'upper right', title = 'Survived')
ax.invert_yaxis()

# Plot 3: Box plot showing Title distribution across Passenger Classes, split by survival status
sns.boxplot(x = 'Pclass', y = 'Title', hue = 'Survived', data = train, ax = axes[2], boxprops=dict(alpha=0.8))
axes[2].set_title('Pclass vs Title Correlation')
ax.legend(loc = 'upper right', title = 'Survived')
ax.invert_yaxis()

plt.tight_layout()
# Display the figure with all three plots
plt.show()

In [ ]:
# Create a figure with 3 subplots in a row, with a size of 12x9 inches
fig, axes = plt.subplots(1, 3, figsize = (12, 9))
# Update the font size for all plot elements to 12
plt.rcParams.update({'font.size': 12})

columns = ['Embarked', 'Pclass', 'AgeBins']

for i, (column, ax) in enumerate(zip(columns, axes.flat), 0):

    sns.barplot(x = 'Sex', y = 'Survived', hue = column, data = train, ax = ax, orient = 'v', palette = 'viridis', alpha = plotalpha)
    # Set the title for each subplot
    ax.set_title(f'Survived by Sex and {column}')

    ax.legend(loc = 'upper left', title = 'Survived')
# Adjust the layout to prevent overlap between subplots
plt.tight_layout()
# Display the figure
plt.show()

In [ ]:
g = sns.FacetGrid(data=train, col='Embarked', legend_out=True)

g.map(sns.lineplot, 'Pclass', 'Survived', 'Sex', palette='mako')
g.add_legend()

# Adjust the layout to prevent overlap between subplots
plt.tight_layout()
plt.show()

In [ ]:
for col in ['Age', 'FamilySize']:

    f = sns.FacetGrid(data = train, hue = 'Survived', aspect = 4)

    # Plot kernel density estimate (KDE) for the current column, filled with color based on survival status

    f.map(sns.kdeplot, col, fill = True, hue = train['Survived'], palette = 'icefire')

    f.set(xlim = ((0 if col == 'Age' else  1), train[col].max()))

    # Add a legend to the plot

    f.add_legend(loc = 'center right')


    sns.move_legend(f, 'center right', bbox_to_anchor=(0.99, 0.8))

    plt.tight_layout()

# Display all created plots

plt.show()

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train = pd.get_dummies(train, dtype = int)
test = pd.get_dummies(test, dtype = int)

train.head()

In [ ]:
plt.figure(figsize = (12, 12))

sns.heatmap(train.corr(), annot = True, fmt = '0.1f', cmap = 'flare')
# Adjust the layout to ensure everything fits properly
plt.tight_layout()
plt.show()

In [ ]:
# Split the data into features and target variables
X_train = train.drop(['Survived'], axis = 1)
X_test = test
y_train = train['Survived']
y_test = gender_sub['Survived']

In [ ]:
model_params = [
               {'model': LogisticRegression(max_iter = 1000),
                'params' : {'l1_ratio': [0],
                            'solver': ['lbfgs'],
                           }},
               {'model': SGDClassifier(),
                 'params': {'loss': ['hinge', 'log_loss'],
                            'penalty': ['l1', 'l2', 'elasticnet']
                                 }},
               {'model': Perceptron(),
                'params': {'penalty': ['l1', 'l2', 'elasticnet']
                              }},
               {'model': SVC(),                                # Support Vector Classifier
                'params': {'kernel': ['rbf', 'linear', 'sigmoid'],
                           'gamma': ['scale', 'auto']
                        }},

               {'model': KNeighborsClassifier(),
                'params': {'n_neighbors' : [5],
                           'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']  # Testing different search algorithms
                          }},
               {'model': GaussianNB(),
                'params': {}
               },
               {'model': MultinomialNB(),
                'params': {}
               },
               {'model': DecisionTreeClassifier(),             # Decision Tree classifier
                'params': {'criterion': ['gini', 'entropy', 'log_loss'],
                           'splitter': ['best', 'random'],
                           'random_state': [42]
                                           }}]
model_params = [dict(i) for i in model_params]

In [ ]:
# Initialize empty lists to store models, model names, scores, and cross-validation scores
models = []
model_names = []
scores = []
crossvalscores = []

# Iterate through each model in model_params
for item in model_params:
    # Get the model from the current item
    model = item['model']

    # Train the model on the training data
    model.fit(X_train, y_train)

    # Calculate the training accuracy score
    score = model.score(X_train, y_train)

        crossvalscore = np.round(cross_val_score(model, X_train, y_train, cv = 20)* 100, 2)

    # Get the class name of the model
    modelclassname = model.__class__.__name__

        models.append(model)
    model_names.append(modelclassname)
    crossvalscores.append(crossvalscore)
    scores.append(score)

    # Print model name and its training accuracy as percentage
    print(modelclassname, ":", round(score* 100, 2), "%")
    print('-' * 40)

    # Print cross-validation scores
    print(f'cross_val_score: {crossvalscore}')
    print('*' * 40)

all_scores = np.concatenate(crossvalscores)
xmin, xmax = all_scores.min(), all_scores.max()

# Create a 2x4 grid of subplots for visualizing cross-validation scores
fig, axes = plt.subplots(2, 4, figsize = (12, 8))
plt.rcParams.update({'font.size': 10})  # Set font size for all plots

for (model_name, score, ax) in (zip(model_names, crossvalscores, axes.flat)):

    mean_score = np.mean(score)

    ax = sns.violinplot(x=score, ax=ax, color='blueviolet', inner = 'quartile')

    ax.axvline(mean_score, color='red', linestyle='--', label=f'Mean: {mean_score:.3f}')

    # Set transparency for the violin plots
    for patch in ax.collections:
        patch.set_alpha(0.8)

    ax.set_xlim(xmin, xmax)

    # Set title and y-axis label
    ax.set_title(f'{model_name} Cross-Val Scores')
    ax.set_ylabel('Folds')

# Turn off any unused subplots
for i in range(len(model_names), len(axes.flat)):
    axes.flat[i].set_axis_off()

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

In [ ]:
all_scores = np.concatenate(crossvalscores)
# Find the minimum and maximum values in the scores
xmin, xmax = all_scores.min(), all_scores.max()
# Get the shape of the combined scores array
all_scores.shape

In [ ]:
# Compute per-model cross-validation summary statistics
mean_scores = [np.mean(scores) for scores in crossvalscores]
std_scores = [np.std(scores) for scores in crossvalscores]

# Aggregate results into a single DataFrame and rank models by mean CV score
scoresdf = (
    pd.DataFrame({
        'model_names': model_names,
        'crossvalscores': crossvalscores,
        'mean_scores': mean_scores,
        'std_scores': std_scores
    })
    .sort_values('mean_scores', ascending=False)
)

# Initialize figure with explicit size for consistent rendering
plt.figure(figsize=(10, 6))

# Plot mean CV scores per model (points only; error handled manually)
sns.pointplot(
    x='model_names',
    y='mean_scores',
    data=scoresdf,
    errorbar=None,
    color='magenta'
)

# Overlay manual error bars using precomputed standard deviations
plt.errorbar(
    x=range(len(scoresdf)),
    y=scoresdf['mean_scores'],
    yerr=scoresdf['std_scores'],
    fmt='none',
    ecolor='black',
    capsize=5
)

plt.xticks(rotation=60)

# Axis labels and descriptive title
plt.xlabel('Model Names')
plt.ylabel('Mean CV Score')
plt.title('Model Cross-Validation Performance (20-fold CV)')

# Prevent label clipping and render plot
plt.tight_layout()
plt.show()


In [ ]:
def featureimportance(model, data):
    # Extract model class name for labeling and reporting
    model_name = model.__class__.__name__

    # Ensure input data provides feature names (e.g., pandas DataFrame)
    if hasattr(data, 'columns'):
        feature_names = data.columns
        values = None
        title = None

        # Tree-based models: use built-in feature importances
        if hasattr(model, 'feature_importances_'):
            values = model.feature_importances_
            title = 'Feature Importance'

        # Linear models: use coefficients (single-output case)
        elif hasattr(model, 'coef_'):
            values = model.coef_[0]
            title = 'Coefficients absolute values'

        # Graceful handling for unsupported models
        else:
            print(f'{model_name} has no feature importance or coefficient')

        return model_name, feature_names, values, title


# Compute feature importance metadata for each configured model
results = list(
    map(lambda item: featureimportance(item['model'], X_train), model_params)
)

# Visualize feature importance per model where available
for model_name, feature_name, value, title in results:
    if value is not None:
        # Rank features by importance
        indices = np.argsort(value)[::-1]

        # Prepare a sorted DataFrame for plotting
        importances_df = (
            pd.DataFrame({
                'Feature Name': feature_name,
                'Feature Importance': value
            })
            .sort_values('Feature Importance', ascending=False)
        )

        # Scale figure height to number of features for readability
        plt.figure(figsize=(10, max(6, len(importances_df) * 0.3)))

        sns.barplot(
            data=importances_df,
            x='Feature Importance',
            y='Feature Name'
        )

        # Model-specific, descriptive title and axis labels
        plt.title(f"{model_name} – {title}")
        plt.xlabel(title)

        # Optimize layout and render
        plt.tight_layout()
        plt.show()

In [ ]:
def TuneHyperParameters(model, params):
    # Initialize grid search with fixed CV strategy and strict error handling
    gridsearch = GridSearchCV(
        model,
        params,
        cv=5,
        error_score='raise'
    )

    # Fit grid search on training data to identify optimal hyperparameters
    gridsearch.fit(X_train, y_train)

    bestparams = gridsearch.best_params_
    bestscores = round(gridsearch.best_score_ * 100, 2)
    bestmodel = gridsearch.best_estimator_

    # Return tuning results as a structured tuple
    return bestparams, bestscores, bestmodel

aftertuning = [
    TuneHyperParameters(item['model'], item['params'])
    for item in model_params
]

for params, score, model in aftertuning:
    print(
        f"{model.__class__.__name__}: {score}% "
        f"------------ and the best params are {params}"
    )

In [ ]:
# Build a summary DataFrame of tuning results with model names as index
bestscores = pd.DataFrame(
    aftertuning,
    index=pd.Series(
        item['model'] for item in model_params
    )
    .astype(str)
    .str.extract(r'([A-Za-z]+)'),
    # Assign explicit, descriptive column names
    columns=['Hyperparams', 'Tuned Accuracy', 'Full Model']
)

# Display aggregated hyperparameter tuning results
bestscores

In [ ]:
# Retain only the tuned performance metric for comparison
tuned = bestscores.drop(['Hyperparams', 'Full Model'], axis=1)

# Reset index to expose model names as an explicit column
tuned_reset = tuned.reset_index().rename(columns={'index': 'Model'})

tuned_reset['Model'] = (
    tuned_reset['Model']
    .astype(str)
    .str.replace(r'[^A-Za-z]', '', regex=True)
)
tuned_reset = tuned_reset.sort_values('Tuned Accuracy', ascending=False)

# Display the final, ordered tuning summary
tuned_reset

In [ ]:
plt.figure(figsize=(8, 6))
sns.pointplot(
    x='Model',
    y='Tuned Accuracy',
    data=tuned_reset,
    color='magenta'
)

plt.xticks(rotation=60)

plt.title('Model Scoring After Tuning')

# Adjust layout to avoid clipping of labels and title
plt.tight_layout()
plt.show()

In [ ]:
# Initialize a hard-voting ensemble using all configured base models
VotingClassifier = VotingClassifier(
    estimators=[
        (item['model'].__class__.__name__, item['model'])
        for item in model_params
    ],
    voting='hard'
)

VotingClassifier.fit(X_train, y_train)

In [ ]:
# Evaluate the trained VotingClassifier on the training set
VotingClassifierScore = VotingClassifier.score(X_train, y_train)

VotingClassifierScore

In [ ]:
voting_y_pred = VotingClassifier.predict(X_test)

In [ ]:
# Compute the accuracy of the VotingClassifier on the test set
test_accuracy = accuracy_score(y_test, voting_y_pred)

# Display test set accuracy
test_accuracy

In [ ]:
# Initialize lists to store evaluation metrics and predictions for each model
predictions = []
accuracy_scores = []
mean_squared_errors = []
classification_reports = []
confusion_matrices = []
roc_auc_scores = []
rocs = []
aucscores = []
precision_recall = []
avg_precisions = []
# Evaluate each model on the test set
for model in models:
    # Compute predicted probabilities if available; fallback to decision function
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_pred_proba = model.decision_function(X_test)

    # Generate class predictions
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    acc_score = round(accuracy_score(y_test, y_pred) * 100, 2)
    mse = mean_squared_error(y_test, y_pred)
    cls_rep = classification_report(y_test, y_pred)
    confusion_mx = confusion_matrix(y_test, y_pred)
    rocauc_score = roc_auc_score(y_test, y_pred)
    precision, recall, thresholds = precision_recall_curve(y_test, y_pred)
    avg_precision = average_precision_score(y_test, y_pred)
    # Compute ROC curve and AUC score
    fpr, tpr, threshold = roc_curve(y_test, y_pred)
    aucscore = auc(fpr, tpr)

    # Append results to corresponding lists
    predictions.append(y_pred)
    accuracy_scores.append(acc_score)
    mean_squared_errors.append(mse)
    classification_reports.append(cls_rep)
    confusion_matrices.append(confusion_mx)
    roc_auc_scores.append(rocauc_score)
    rocs.append([fpr, tpr, threshold])
    aucscores.append(aucscore)
    precision_recall.append([precision, recall, thresholds])
    avg_precisions.append(avg_precision)

    # Print a summary of metrics for the current model
    print('Model:', model.__class__.__name__.center(50))
    print('Accuracy:', acc_score)
    print('MSE:', mse)
    print('Classification Report: \n', cls_rep)
    print('Confusion Matrix: \n', confusion_mx)
    print('ROC_AUC:', rocauc_score)
    print('ROC Curve:\n', fpr, tpr, threshold)
    print('AUC:', aucscore)
    print('Precision_Recall: \n', precision, recall, thresholds)
    print('Average Precision:', avg_precision)
    print('_' * 60)

In [ ]:
# Extract model names from the index of bestscores DataFrame
ourmodelnames = list(bestscores.index)

# Clean up names by stripping extra characters like parentheses or quotes
ourmodelnames = [str(i).strip("(),'") for i in ourmodelnames]


In [ ]:
# Aggregate all evaluation metrics into a single DataFrame for easy comparison
metricsdf = pd.DataFrame({
    'Models': ourmodelnames,
    'Accuracy': accuracy_scores,
    'MSE': mean_squared_errors,
    'Confusion Matrix': confusion_matrices,
    'ROC-AUC': roc_auc_scores,
    'ROC Curve': rocs,
    'AUC': aucscores,
    'Precision-Recall': precision_recall,
    'Avg Precision': avg_precisions
})

# Display the compiled metrics for all models
metricsdf

In [ ]:
# Set up figure size and font scaling for readability
plt.figure(figsize=(10, 6))
plt.rcParams.update({'font.size': 12})

# Prepare the sorted data
sorted_df = metricsdf.sort_values('Accuracy', ascending=False)
# Plot accuracy scores for all models in descending order
plot = sns.pointplot(
    x='Models',
    y='Accuracy',
    data=metricsdf.sort_values('Accuracy', ascending=False),
    color='tab:olive'
)
# We loop through the sorted accuracy values and their index positions
for i, value in enumerate(sorted_df['Accuracy']):
    plot.text(
        i,
        value + 0.01,
        f'{value:.2f}',
        ha='center',
        va='bottom',
        fontweight='bold'
    )
# Label axes and add descriptive title
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Models Accuracy Scores for Predicted Data')

# Rotate x-axis labels to prevent overlap
plt.xticks(rotation=50)

# Adjust layout to prevent clipping of labels and title
plt.tight_layout()

# Render the plot
plt.show()

In [ ]:
# Initialize lists to store evaluation metrics for tuned models
predictions_tuning = []
accuracy_scores_tuning = []
mean_squared_errors_tuning = []
classification_reports_tuning = []
confusion_matrices_tuning = []
roc_auc_scores_tuning = []
rocs_tuning = []
aucscores_tuning = []
precision_recall_tuning = []
avg_precisions_tuning = []

def extract_model(obj):
    # Directly return if object has a predict method
    if hasattr(obj, "predict"):
        return obj

    # Handle dictionaries containing model objects
    if isinstance(obj, dict):
        for key in ['model', 'estimator', 'best_estimator']:
            if key in obj and hasattr(obj[key], "predict"):
                return obj[key]

    if isinstance(obj, tuple):
        for element in obj:
            if hasattr(element, "predict"):
                return element

    return None

# Evaluate each tuned model
for item in aftertuning:
    model = extract_model(item)
    if model is None:
        print(f"Skipping object (no estimator found): {type(item)}")
        continue

    # Predicted probabilities or decision function (for ROC)
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_pred_proba = model.decision_function(X_test)

    # Generate class predictions
    y_pred = model.predict(X_test)

    # Compute evaluation metrics
    acc = round(accuracy_score(y_test, y_pred) * 100, 2)
    mse = mean_squared_error(y_test, y_pred)
    cls_rep = classification_report(y_test, y_pred)
    conf_mx = confusion_matrix(y_test, y_pred)
    tuned_precision, tuned_recall, tuned_thresholds = precision_recall_curve(y_test, y_pred)
    avg_precision = average_precision_score(y_test, y_pred)

    # Compute ROC AUC only if valid for the data
    try:
        rocauc = roc_auc_score(y_test, y_pred)
        fpr, tpr, threshold = roc_curve(y_test, y_pred)
        aucscore = auc(fpr, tpr)
    except ValueError:
        rocauc = None
        fpr, tpr, threshold = None, None, None
        aucscore = None

    # Append metrics to respective lists
    predictions_tuning.append(y_pred)
    accuracy_scores_tuning.append(acc)
    mean_squared_errors_tuning.append(mse)
    classification_reports_tuning.append(cls_rep)
    confusion_matrices_tuning.append(conf_mx)
    roc_auc_scores_tuning.append(rocauc)
    rocs_tuning.append([fpr, tpr, threshold])
    aucscores_tuning.append(aucscore)
    precision_recall_tuning.append([precision, recall, thresholds])
    avg_precisions_tuning.append(avg_precision)

    # Print a summary for each model
    print('Model:', model.__class__.__name__.center(50))
    print('Accuracy:', acc)
    print('MSE:', mse)
    print('Classification Report: \n', cls_rep)
    print('Confusion Matrix: \n', conf_mx)
    print('ROC_AUC:', rocauc)
    print('ROC Curve:\n', fpr, tpr, threshold)
    print('AUC:', aucscore)
    print('Precision_Recall: \n', precision, recall, thresholds )
    print('Average Precision:', avg_precision)
    print('_' * 60)

In [ ]:
# Compile evaluation metrics for all tuned models into a single DataFrame
tunedmetricsdf = pd.DataFrame({
    'Models': ourmodelnames,
    'Accuracy': accuracy_scores_tuning,
    'MSE': mean_squared_errors_tuning,
    'Confusion Matrix': confusion_matrices_tuning,
    'ROC-AUC': roc_auc_scores_tuning,
    'ROC Curve': rocs_tuning,
    'AUC': aucscores_tuning,
    'Precision-Recall': precision_recall_tuning,
    'Avg Precision': avg_precisions_tuning
})

# Display the metrics DataFrame for all tuned models
tunedmetricsdf

In [ ]:
# Set up figure size and font scaling for readability
plt.figure(figsize=(10, 6))
plt.rcParams.update({'font.size': 12})
ax = sns.pointplot(
    x='Models',
    y='Accuracy',
    data=tunedmetricsdf.sort_values('Accuracy', ascending=False),
    color='tab:cyan'
)
# Loop to add the actual score on top of each point
sorted_data = tunedmetricsdf.sort_values('Accuracy', ascending=False)
for i, score in enumerate(sorted_data['Accuracy']):
    ax.text(i, score + 0.005, f'{score:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Tuned Models Accuracy Scores for Predicted Data')

# Rotate x-axis labels to prevent overlap
plt.xticks(rotation=40)

plt.tight_layout()

# Render the plot
plt.show()

In [ ]:
comparison_df = pd.concat([
    metricsdf.assign(Type='Baseline'),
    tunedmetricsdf.assign(Type='Tuned')
])

# 2. Set up the visual style
plt.figure(figsize=(12, 6))
# sns.set_style("whitegrid")

# 3. Create the comparison plot
ax = sns.pointplot(
    x= 'Models',
    y= 'Accuracy',
    hue= 'Type',
    data= comparison_df,
    palette= {'Baseline': 'tab:olive', 'Tuned': 'tab:cyan'},
    markers= ["o", "s"],
    linestyles= ["--", "-"]
)

# 4. Polish the chart
plt.title('Performance Comparison: Baseline vs. Tuned Models', fontsize=16)
plt.ylabel('Accuracy Score')
plt.xticks(rotation=45)
plt.legend(title='Stage')
plt.tight_layout()

plt.show()

In [ ]:
# Initialize ROC plot
plt.figure()

plt.plot([0, 1], [0, 1], lw=2, color='navy', linestyle='--')

# Plot ROC curves for each tuned model
for _, row in tunedmetricsdf.iterrows():
    fpr, tpr, _ = row['ROC Curve']
    roc_auc = row['ROC-AUC']
    if fpr is not None and tpr is not None:
        plt.plot(
            fpr,
            tpr,
            lw=2,
            label=f"{row['Models']} (AUC = {roc_auc:.2f})"
        )

# Repeat diagonal for clarity
plt.plot([0, 1], [0, 1], lw=2, color='navy', linestyle='--')

# Set axis limits for clarity
plt.xlim([-0.02, 1.0])
plt.ylim([0.0, 1.05])

# Add labels and title
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')

# Add legend with smaller font to accommodate multiple models
plt.legend(loc="lower right", fontsize=9)

# Adjust layout to prevent clipping
plt.tight_layout()

# Render the ROC plot
plt.show()

In [ ]:
# Select the confusion matrix of the model with the highest accuracy
mnbconfusionmx = confusion_matrices[accuracy_scores.index(max(accuracy_scores))]

# Initialize figure with a compact size
plt.figure(figsize=(4, 3))

# Plot the confusion matrix as a heatmap with annotations
sns.heatmap(
    mnbconfusionmx,
    annot=True,
    fmt='0.1f',       # Format numbers to 1 decimal place
    cmap='flare')      # Color map for visual emphasis
# Labeling for clarity
plt.xlabel('Predicted Label', fontweight='bold')
plt.ylabel('Actual Label', fontweight='bold')
plt.title(f'Confusion Matrix: {tunedmetricsdf.iloc[accuracy_scores.index(max(accuracy_scores))]["Models"]}', fontsize=14)

# Adjust layout to prevent clipping of labels
plt.tight_layout()

# Render the heatmap
plt.show()

In [ ]:
# Identify the ROC curve and AUC of the model with the highest accuracy
mnbroccurve = rocs[accuracy_scores.index(max(accuracy_scores))]
false_positive_ratio, true_positive_ratio, threshold = mnbroccurve

# Retrieve corresponding AUC scores
mnbauc = aucscores[accuracy_scores.index(max(accuracy_scores))]
mnbrocauc = roc_auc_scores[accuracy_scores.index(max(accuracy_scores))]

# Display ROC curve data and AUC values
mnbroccurve, mnbauc, mnbrocauc

In [ ]:
plt.figure()

# Plot the ROC curve
plt.plot(
    false_positive_ratio,
    true_positive_ratio,
    color='darkorange',
    lw=2,
    label=f'ROC curve (AUC = {mnbrocauc:.2f})'
)

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

# Set axis limits
plt.xlim([-0.05, 1.0])
plt.ylim([0.0, 1.05])

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')

plt.legend(loc="lower right")

plt.tight_layout()

# Render the ROC plot
plt.show()

In [ ]:
# Get the precision, recall, and thresholds from the best model based on accuracy
precisionbest, recallbest, thresholdsbest = precision_recall[accuracy_scores.index(max(accuracy_scores))]

# Store the average precision of the best model
avgprecisionbestmodel = avg_precisions[accuracy_scores.index(max(accuracy_scores))]

# 2. Plot the Precision-Recall Curve
plt.figure(figsize=(8, 6))
plt.plot(recallbest, precisionbest, color='limegreen', lw=2, label=f'PR Curve (AP = {avg_precision:.2f})')

baseline = sum(y_test == 1) / len(y_test)
plt.axhline(y=baseline, color='navy', linestyle='--', label=f'Baseline ({baseline:.2f})')

# 4. Formatting the plot with labels, title, legend, and grid
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Focus on Positive Class)')
plt.legend(bbox_to_anchor=(0.95, 0.95), loc="upper right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()  # Display the plot

In [ ]:
# Select the model with the highest accuracy from the trained models
thebestmodel = models[accuracy_scores.index(max(accuracy_scores))]

# Display the best-performing model
thebestmodel


In [ ]:
prediction = thebestmodel.predict(X_test)

# `prediction` now contains the predicted class labels for the test data

In [ ]:
# Create a submission DataFrame with PassengerId and predicted survival
gender_submission = pd.DataFrame({
    'PassengerId': gender_sub['PassengerId'],
    'Survived': prediction
})

# Export the DataFrame to CSV for submission
gender_submission.to_csv('my submission.csv', index=False)